<a href="https://colab.research.google.com/github/selim679/databricks-streaming-pipeline/blob/main/03_gold_layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql.functions import *

df_silver = spark.read.format("delta").load("/Volumes/workspace/default/youtubeseries/delta/silver")
print(f"Silver table row count: {df_silver.count()}")
display(df_silver)

df_daily = df_silver.groupBy("event_date").agg(
    count("*").alias("total_events"),
    sum("amount").alias("total_revenue"),
    countDistinct("user_id").alias("active_users")
)

df_event_type = df_silver.groupBy("event_date", "event_type").agg(
    count("*").alias("event_count"),
    sum("amount").alias("revenue")
)

df_daily.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("event_date") \
    .save("/Volumes/workspace/default/youtubeseries/delta/gold/daily_metrics")

df_event_type.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("event_date") \
    .save("/Volumes/workspace/default/youtubeseries/delta/gold/event_metrics")

In [ ]:
display(spark.read.format("delta").load("/Volumes/workspace/default/youtubeseries/delta/gold/daily_metrics"))

In [ ]:
df_conversion = df_silver.groupBy("event_date").agg(
    (sum(when(col("event_type") == "purchase", 1).otherwise(0)) /
     sum(when(col("event_type") == "view", 1).otherwise(0))
    ).alias("conversion_rate")
)

In [ ]:
df_daily = spark.read.format("delta").load("/Volumes/workspace/default/youtubeseries/delta/gold/daily_metrics")

df_event = spark.read.format("delta").load("/Volumes/workspace/default/youtubeseries/delta/gold/event_metrics")

display(df_daily)

In [ ]:
import plotly.express as px

df_pandas = df_daily.toPandas()
fig = px.line(df_pandas,
              x='event_date',
              y='total_revenue',
              title='Revenue Evolution Over Time',
              labels={'total_revenue': 'Total Revenue ($)', 'event_date': 'Date'})
fig.show()

In [ ]:
display(df_daily)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df_pandas = df_daily.orderBy("event_date").toPandas()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Revenue Over Time', 'Active Users Over Time')
)

fig.add_trace(
    go.Scatter(x=df_pandas['event_date'], y=df_pandas['total_revenue'],
               mode='lines+markers', name='Revenue'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=df_pandas['event_date'], y=df_pandas['active_users'],
           name='Active Users'),
    row=1, col=2
)

fig.update_layout(height=400, showlegend=False, title_text="Daily Metrics Dashboard")
fig.update_xaxes(title_text="Date", row=1, col=1)
fig.update_xaxes(title_text="Date", row=1, col=2)
fig.update_yaxes(title_text="Revenue ($)", row=1, col=1)
fig.update_yaxes(title_text="Users", row=1, col=2)

displayHTML(fig.to_html())

In [ ]:
display(df_event)

In [ ]:
import plotly.express as px

df_event_summary = df_event.groupBy("event_type").agg(
    sum("event_count").alias("total_events"),
    sum("revenue").alias("total_revenue")
)
df_event_pandas = df_event_summary.orderBy("total_events", ascending=False).toPandas()

fig = px.bar(df_event_pandas,
             x='event_type',
             y='total_events',
             title='Event Distribution by Type',
             labels={'total_events': 'Total Events', 'event_type': 'Event Type'},
             color='event_type')

displayHTML(fig.to_html())